# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print("\nDataset @id:", metadata['@id'])
print("Conforms To:", metadata['conformsTo'])
print("Version:", metadata['version'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll inspect the Croissant metadata to identify the available record sets (`cr:RecordSet`), their `@id`, and the fields inside each record set.

In [ ]:
# Examine record sets and their fields by @id

# Fetch the raw JSON-LD metadata
dataset_jsonld = dataset.metadata.to_jsonld()

# Helper function to extract record set @ids and provide field info
def extract_recordsets_and_fields(dataset_jsonld):
    record_sets = []
    record_set_fields = {}
    for obj in dataset_jsonld:
        if obj.get('@type') == 'cr:RecordSet':
            rs_id = obj['@id']
            record_sets.append(rs_id)
            # Gather fields for this record set
            fields = []
            for field_ref in obj.get('cr:field', []):
                fields.append(field_ref['@id'])
            record_set_fields[rs_id] = fields
    return record_sets, record_set_fields

record_sets, record_set_fields = extract_recordsets_and_fields(dataset_jsonld)

print("Available record sets (@id):")
for rs in record_sets:
    print(f"- {rs}")

print("\nFields per record set (@id):")
for rs in record_sets:
    print(f"Record set {rs}: fields:")
    for f in record_set_fields[rs]:
        print(f"  - {f}")

In [ ]:
# You can print the first few records from each record set using their @id
for record_set_id in record_sets:
    print(f"\nFirst record from record set: {record_set_id}")
    # generator returns dicts with keys as field @id
    try:
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            print(record)
            if i >= 0:
                break
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the `@id` of each record set and fields as keys.

In [ ]:
# Extract data from each record set

dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame for record set {record_set_id} loaded: shape {df.shape}")
        print(f"Columns (@id): {df.columns.tolist()}\n")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load DataFrame for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Choose one record set and numeric field for demonstration. 

Referencing everything by `@id`, we select the first record set and inspect its fields for numeric column(s).

In [ ]:
# Pick a record set and a numeric field via their @id

# Example: Use first record set and first numeric field
selected_record_set_id = record_sets[0] if record_sets else None
selected_df = dataframes.get(selected_record_set_id)

if selected_df is not None:
    # Find numeric fields (float/int) - assume float/int detected in dataframe automatically
    numeric_fields = [col for col in selected_df.columns if pd.api.types.is_numeric_dtype(selected_df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field (@id): {numeric_field_id}")
    else:
        print("No numeric fields found.")

    # Demonstrate filtering and normalization
    threshold = 10
    filtered_df = selected_df[selected_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by another field if available
    group_field_candidates = [col for col in selected_df.columns if col != numeric_field_id]
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll generate basic histograms and scatter plots using the chosen numeric and grouping fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_df is not None and numeric_fields:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(selected_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot vs group_field
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.scatterplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} vs {group_field_id} (filtered)")
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.show()

## 6. Conclusion
We have loaded and explored the FAIR^2 Mass Spectrometry dataset using its Croissant schema and the `mlcroissant` library. By referencing all entities via their `@id`, we ensured robust and reproducible data access. The notebook demonstrates:
- Dataset metadata overview
- Listing available record sets and fields with their `@id`
- Dynamic extraction of tabular data from record sets
- Basic filtering, normalization, and grouping on numeric fields
- Visualizing the distributions and relationships in the data

This process can now be adapted for further analyses, feature engineering, or model building using the standardized Croissant schema and `mlcroissant` tools.